# 🎯 Parte 2 — De pontuações para porcentagens: a função *softmax*

Na Parte 1 vimos que o **produto escalar** dá um número que mostra o quanto duas palavras combinam.

Agora vamos responder:

> **Se uma palavra pudesse "prestar atenção" nas outras da frase, quanto (em %) ela olharia para cada uma?**

📌 Execute as células na ordem com `Shift + Enter`.

## Preparação — trazendo o que fizemos na Parte 1

Cada notebook do Colab é independente, então precisamos criar de novo os embeddings e a função de produto escalar.

In [ ]:
import math   # biblioteca de matemática do Python (vamos usar só math.exp)

#                [animal, comida, ação]
embeddings = {
    "o":         [0.1,    0.1,    0.1],
    "gato":      [0.9,    0.1,    0.0],
    "cachorro":  [0.8,    0.1,    0.1],
    "leite":     [0.0,    0.9,    0.1],
    "queijo":    [0.1,    0.8,    0.0],
    "bebeu":     [0.1,    0.3,    0.9],
    "comeu":     [0.1,    0.4,    0.8],
}

def produto_escalar(vetor_a, vetor_b):
    soma = 0
    for i in range(len(vetor_a)):
        soma = soma + vetor_a[i] * vetor_b[i]
    return round(soma, 2)

print("Tudo pronto!")

## Passo 1 — Pontuações (escores) de atenção

Vamos usar a frase **"gato bebeu leite"** e escolher a palavra **"bebeu"** como foco.

Pergunta: *para entender "bebeu", em quem eu devo prestar atenção?*

Calculamos o produto escalar de "bebeu" com **cada palavra da frase** (inclusive com ela mesma). Esses números são chamados de **escores**.

In [ ]:
frase = ["gato", "bebeu", "leite"]
foco = "bebeu"

escores = []
for palavra in frase:
    valor = produto_escalar(embeddings[foco], embeddings[palavra])
    escores.append(valor)
    print(foco, "olhando para", palavra, "-> escore =", valor)

print()
print("Escores:", escores)

## Passo 2 — Transformando em porcentagens (tentativa 1)

Escores soltos são difíceis de interpretar. Queremos **porcentagens que somem 100%**.

A ideia mais simples: **dividir cada escore pela soma de todos**.

In [ ]:
total = 0
for valor in escores:
    total = total + valor

print("Soma dos escores:", round(total, 2))
for i in range(len(frase)):
    porcentagem = escores[i] / total * 100
    print(frase[i], "->", round(porcentagem, 1), "%")

Parece funcionar... mas e se algum escore for **negativo**? (Nos modelos reais, isso acontece o tempo todo!)

In [ ]:
escores_teste = [2.0, -1.0, 0.5]

total = 0
for valor in escores_teste:
    total = total + valor

for valor in escores_teste:
    print("escore", valor, "->", round(valor / total * 100, 1), "%")

😱 Apareceu uma **porcentagem negativa** e outra **maior que 100%**! Isso não faz sentido.

## Passo 3 — O truque da exponencial

A função **exponencial** (`math.exp`) tem duas propriedades perfeitas para nós:

1. O resultado é **sempre positivo**, mesmo para números negativos;
2. Números maiores ficam **muito** maiores (ela "destaca o vencedor").

In [ ]:
for x in [-2, -1, 0, 1, 2, 3]:
    print("exp(", x, ") =", round(math.exp(x), 3))

## Passo 4 — A função **softmax**, passo a passo

O softmax é só isso:

1. Aplicar `exp` em cada escore;
2. Somar os resultados;
3. Dividir cada um pela soma.

In [ ]:
escores_teste = [2.0, -1.0, 0.5]

# 1) exponencial de cada escore
exponenciais = []
for valor in escores_teste:
    exponenciais.append(math.exp(valor))
print("1) Exponenciais:", exponenciais)

# 2) soma
total = 0
for valor in exponenciais:
    total = total + valor
print("2) Soma:", round(total, 3))

# 3) divisão
pesos = []
for valor in exponenciais:
    pesos.append(valor / total)
print("3) Pesos:")
for i in range(len(escores_teste)):
    print("   escore", escores_teste[i], "->", round(pesos[i] * 100, 1), "%")

✅ Agora tudo é positivo e soma 100%. Vamos guardar isso em uma função:

In [ ]:
def softmax(lista_de_escores):
    exponenciais = []
    for valor in lista_de_escores:
        exponenciais.append(math.exp(valor))

    total = 0
    for valor in exponenciais:
        total = total + valor

    pesos = []
    for valor in exponenciais:
        pesos.append(round(valor / total, 3))
    return pesos

print(softmax([2.0, -1.0, 0.5]))

## Passo 5 — Visualizando a atenção com barrinhas

Vamos criar uma função que desenha uma barra com `█` proporcional à porcentagem.

In [ ]:
def mostrar_barras(palavras, pesos):
    for i in range(len(palavras)):
        porcentagem = pesos[i] * 100
        barra = "█" * int(porcentagem / 2)   # 1 bloco a cada 2%
        print(palavras[i].ljust(10), barra, round(porcentagem, 1), "%")

frase = ["gato", "bebeu", "leite"]
foco = "bebeu"

escores = []
for palavra in frase:
    escores.append(produto_escalar(embeddings[foco], embeddings[palavra]))

pesos = softmax(escores)
print("Atenção da palavra '" + foco + "':")
mostrar_barras(frase, pesos)

## Passo 6 — A tabela de atenção completa

No Transformer, **todas as palavras olham para todas as palavras ao mesmo tempo**. Vamos repetir o processo para cada palavra da frase:

In [ ]:
frase = ["gato", "bebeu", "leite"]

for foco in frase:
    escores = []
    for palavra in frase:
        escores.append(produto_escalar(embeddings[foco], embeddings[palavra]))
    pesos = softmax(escores)
    print("Atenção de '" + foco + "':")
    mostrar_barras(frase, pesos)
    print()

🔎 **Observe duas coisas:**

1. **Cada palavra presta mais atenção nela mesma.** Faz sentido (uma palavra é sempre "igual" a ela mesma), mas não ajuda muito a entender o contexto. Na Parte 3 vamos resolver isso com **Query** e **Key**.
2. **A atenção está meio "indecisa"**: nenhuma palavra passa muito de 50%. Por quê? Porque nossos escores são números pequenos (entre 0 e 1), e números próximos geram porcentagens próximas.

Veja o que acontece se os escores forem **maiores** (multiplicados por 10):

In [ ]:
foco = "bebeu"
escores = []
for palavra in frase:
    escores.append(produto_escalar(embeddings[foco], embeddings[palavra]))

escores_x10 = []
for valor in escores:
    escores_x10.append(round(valor * 10, 2))

print("Escores normais:", escores)
mostrar_barras(frase, softmax(escores))
print()
print("Escores x 10:", escores_x10)
mostrar_barras(frase, softmax(escores_x10))

💡 Com escores maiores, o softmax fica **decidido** (quase tudo em uma palavra). Com escores menores, fica **indeciso**.
Os Transformers controlam isso **dividindo os escores** por um número — vamos ver isso na Parte 3!

---
## ✏️ Exercício

1. Monte a frase `["cachorro", "comeu", "queijo"]` e mostre a tabela de atenção completa.
2. Teste multiplicar os escores por **0.1** e por **100**. O que acontece com as barras?
3. Desafio: o que acontece com o softmax se **todos os escores forem iguais**? Teste com `softmax([1, 1, 1, 1])`.

In [ ]:
# Complete aqui o exercício
frase = ["cachorro", "comeu", "queijo"]

---
## 📝 Resumo da Parte 2

| Conceito | O que é |
|---|---|
| **Escore** | Produto escalar entre a palavra em foco e outra palavra |
| **Exponencial** | Deixa todo número positivo e destaca os maiores |
| **Softmax** | Transforma escores em **pesos** que somam 100% |
| **Peso de atenção** | Quanto (%) uma palavra "olha" para outra |

➡️ **Na Parte 3** vamos usar esses pesos para **misturar** as palavras e criar o **mecanismo de atenção** completo, com **Query, Key e Value**.